In [1]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import make_pipeline

from pathlib import Path

import sys
sys.path.append("../src")
from time_utils import TimeCleaner
from gap_splitter import LargeGapSplitter

In [2]:
# Display options
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)
DATA_PATH = Path("../Data/Original_Data/EnergyParameterDataset.xlsx")
data = pd.read_excel(DATA_PATH)

In [3]:
class MotorTypeFilter(BaseEstimator, TransformerMixin):
    def __init__(self, motor_map):
        self.motor_map = motor_map

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        dfs = []
        for name, types in self.motor_map.items():
            df = X[X["Type"].isin(types)].copy()
            df["TYPE"] = name
            dfs.append(df)
        return pd.concat(dfs, axis=0)


In [4]:
class RawPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, gap_threshold=1.0):
        self.gap_threshold = gap_threshold

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()

        df = df.drop_duplicates()

        df = TimeCleaner(df).clean()
        df["Time"] = pd.to_datetime(df["Time"])
        df = df.set_index("Time").sort_index()
        df.index.name = "Time"

        df["KWH_diff"] = df["TOTAL_NET_KWH"].diff()
        df["KWH_diff"] = df["KWH_diff"].where(df["KWH_diff"] >= 0)
        df = df.dropna(subset=["KWH_diff"])

        df = LargeGapSplitter(df, threshold_hours=self.gap_threshold).run()

        return df


In [5]:
class HourlyResampler(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = (
            X.resample("1H")
             .agg(
                 HOURLY_KWH=("KWH_diff", "sum"),
                 AVG_CURRENT=("AVG_CURRENT", "mean"),
                 AVG_V_LN=("AVG_V_LN", "mean"),
             )
             .sort_index()
        )
        return df


In [6]:
class MedianImputer(BaseEstimator, TransformerMixin):
    def __init__(self, cols):
        self.cols = cols
        self.medians_ = {}

    def fit(self, X, y=None):
        for col in self.cols:
            self.medians_[col] = X[col].median()
        return self

    def transform(self, X):
        df = X.copy()
        for col in self.cols:
            df[col] = df[col].fillna(self.medians_[col])
        return df


In [7]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()

        df["power_proxy"] = df["AVG_CURRENT"] * df["AVG_V_LN"]

        df["hour"] = df.index.hour
        df["weekday"] = df.index.weekday

        df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
        df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

        df["weekday_sin"] = np.sin(2 * np.pi * df["weekday"] / 7)
        df["weekday_cos"] = np.cos(2 * np.pi * df["weekday"] / 7)

        for lag in [24, 168]:
            df[f"kwh_lag_{lag}"] = df["HOURLY_KWH"].shift(lag)

        df["kwh_roll_2"] = df["HOURLY_KWH"].rolling(2).mean()
        df["kwh_roll_24"] = df["HOURLY_KWH"].rolling(24).mean()

        df["long_gap_flag"] = df["AVG_CURRENT"].isna() & df["AVG_V_LN"].isna()
        gap_id = df["long_gap_flag"].cumsum()
        df["time_since_gap"] = df.groupby(gap_id).cumcount()
        df["time_since_gap_log"] = np.log1p(df["time_since_gap"])

        baseline = df["HOURLY_KWH"].rolling(24, min_periods=12).mean()
        df["low_activity_detected"] = (df["HOURLY_KWH"] < 0.25 * baseline).astype(int)

        rolling_median = df["HOURLY_KWH"].rolling(24, min_periods=12).median()
        mad = (
            (df["HOURLY_KWH"] - rolling_median)
            .abs()
            .rolling(24, min_periods=12)
            .median()
        )

        df["spike_detected"] = (
            df["HOURLY_KWH"] > (rolling_median + 3 * mad)
        ).astype(int)

        return df


In [8]:
class FinalCleaner(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.dropna()


In [9]:
MOTOR_MAP = {
    "YWNC2_CONE": ["YWNC-203", "YWNC2 CONE"],
    "YWNC2_CUP":  ["YWNC-205", "YWNC2 CUP"],
    "YWNC3_CONE": ["YWNC-303", "YWNC3 CONE"],
    "YWNC3_CUP":  ["YWNC-305", "YWNC3 CUP"],
}

energy_pipeline = make_pipeline(
    MotorTypeFilter(MOTOR_MAP),
    RawPreprocessor(gap_threshold=1.0),
    HourlyResampler(),
    MedianImputer(cols=["AVG_CURRENT", "AVG_V_LN"]),
    FeatureEngineer(),
    FinalCleaner()
)


In [10]:
final_df = energy_pipeline.fit_transform(data)

InvalidIndexError: Reindexing only valid with uniquely valued Index objects

In [ ]:
final_df.to_csv("../Data/HourlyTransformedData/hourly_energy_all_types.csv")
final_df.to_pickle("../Data/HourlyTransformedData/hourly_energy_all_types.pkl")


YWNC2_CONE:- 
 HOURLY_KWH     0
AVG_CURRENT    0
AVG_V_LN       0
dtype: int64

YWNC2_CUP:- 
 HOURLY_KWH     0
AVG_CURRENT    0
AVG_V_LN       0
dtype: int64

YWNC3_CONE:- 
 HOURLY_KWH     0
AVG_CURRENT    0
AVG_V_LN       0
dtype: int64

YWNC3_CUP:- 
 HOURLY_KWH     0
AVG_CURRENT    0
AVG_V_LN       0
dtype: int64

